# Bootstrap confidence intervals and significance testing for scaffold metrics

This notebook runs bootstrap sampling **from large output sets** (e.g., millions of molecules) by repeatedly
drawing a subsample (default: **250,000 molecules**) with replacement and computing three scaffold-based metrics:

- **SED**: unique scaffolds in the OS / cOS  
- **ASER**: Active scaffolds in the OS / cOS
- **TUPOR**: Unique Active scaffolds in the OS / Unique Active scaffolds in the RS  

For each *(receptor, split, scaffold type, generator)* combination we compute:
- the bootstrap mean
- the **(1−α) confidence interval** (default α = 0.05 → 95% CI)

We also save the **bootstrap distributions** per job so we can test **pairwise differences**
between generators (bootstrap CI for Δ and an empirical two-sided p-value).


In [14]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Local module (cleaned + configurable version of your original script)
from src  import bootstrap_scaffold_ci as bci
import importlib as imp
imp.reload(bci)

<module 'src.bootstrap_scaffold_ci' from '/home/filv/phd_projects/iga_2023/git_reccal/new/recall_metrics/src/bootstrap_scaffold_ci.py'>

## 1) Configuration

Two independent parallelism knobs:

- `job_workers`: how many (receptor × split × scaffold × generator) jobs run in parallel  
- `scaffold_workers`: how many processes are used to convert SMILES → scaffolds inside each job  

Tip:
- First run (building caches): `job_workers=1`, `scaffold_workers=CPU-1`, `show_progress=True`
- Later runs (cache hits): `job_workers=CPU-1`, `scaffold_workers=1`, `show_progress=False`


In [15]:
# ---- paths ----
DATA_FOLDER = "../"          # must contain a 'data/' directory as described in the module docstring
RESULTS_DIR = "results_bootstrap_ci"  # will be created if missing

# ---- bootstrap parameters ----
CLUSTERS = [0, 1, 2, 3, 4]
N_SUBSAMPLE = 250_000
N_BOOTSTRAP = 300
ALPHA = 0.05

# ---- parallelism ----
CPU = 170  #os.cpu_count() or 2
job_workers = max(1, CPU - 1)
scaffold_workers = max(1, CPU - 1)

# ---- performance / UX ----
use_cache = True
show_progress = True     # tqdm bars for scaffold conversion
chunksize = 2000         # multiprocessing imap chunksize

cfg = bci.BootstrapConfig(
    clusters=CLUSTERS,
    n_subsample=N_SUBSAMPLE,
    n_bootstrap=N_BOOTSTRAP,
    alpha=ALPHA,
    results_dir=RESULTS_DIR,
    data_folder=DATA_FOLDER,
    job_workers=job_workers,
    scaffold_workers=scaffold_workers,
    chunksize=chunksize,
    use_cache=use_cache,
    show_progress=show_progress,
    save_bootstrap_samples=True,  # needed for significance tests below
)

cfg


BootstrapConfig(clusters=[0, 1, 2, 3, 4], n_subsample=250000, n_bootstrap=300, alpha=0.05, results_dir='results_bootstrap_ci', data_folder='../', job_workers=169, scaffold_workers=169, chunksize=2000, use_cache=True, show_progress=True, save_bootstrap_samples=True)

## 2) Run bootstrap CIs for all jobs

This will write:

- Per job: `.../<generator>/bootstrap_ci.csv` and `.../<generator>/bootstrap_samples/<metric>.npy`
- Global summary CSV: `results_bootstrap_ci/bootstrap_ci_scaffold_metrics_summary.csv`


In [16]:
receptors = ["Glucocorticoid_receptor", "Leukocyte_elastase"]
splits = ["dis", "sim"]
scaffolds = ["csk", "murcko"]
generators = [
    "Molpher",
    "DrugEx_GT_epsilon_0.1",
    "DrugEx_GT_epsilon_0.6",
    "DrugEx_RNN_epsilon_0.1",
    "DrugEx_RNN_epsilon_0.6",
    "GB_GA_log_p_mut_r_0.01",
    "GB_GA_log_p_mut_r_0.5",
    "GB_GA_mut_r_0.01",
    "GB_GA_mut_r_0.5",
    "REINVENT",
    "addcarbon",
]

df_summary = bci.run_all_jobs(receptors, splits, scaffolds, generators, cfg)
df_summary.head()


Jobs: 100%|██████████| 88/88 [09:34<00:00,  6.53s/job]  


[DONE] Wrote: results_bootstrap_ci/bootstrap_ci_scaffold_metrics_summary.csv (9.59 min)


,receptor,split,scaffold,generator,metric,mean,ci_low,ci_high,clusters,error
0,Leukocyte_elastase,dis,csk,GB_GA_log_p_mut_r_0.5,RS,0.326463,0.301616,0.353162,"0,1,2,3,4",None
1,Leukocyte_elastase,dis,csk,GB_GA_log_p_mut_r_0.5,SED,0.073629,0.073341,0.073930,"0,1,2,3,4",None
2,Leukocyte_elastase,dis,csk,GB_GA_log_p_mut_r_0.5,ASER,0.014567,0.014379,0.014747,"0,1,2,3,4",None
3,Glucocorticoid_receptor,dis,csk,addcarbon,RS,0.033403,0.033403,0.033403,"0,1,2,3,4",None
4,Glucocorticoid_receptor,dis,csk,addcarbon,SED,0.003427,0.003425,0.003427,"0,1,2,3,4",None


## 3) Final results table (mean and CI)

The global summary is the main artifact you typically want to use for plots and reporting.


In [17]:
summary_csv = Path(RESULTS_DIR) / "bootstrap_ci_scaffold_metrics_summary.csv"
df = pd.read_csv(summary_csv)

# Drop potential error rows (if some jobs had missing files)
if "error" in df.columns:
    df = df[df["error"].isna()].copy()

# Nice formatting
df["CI"] = df.apply(lambda r: f"[{r['ci_low']:.4f}, {r['ci_high']:.4f}]", axis=1)
df_out = df[["receptor", "split", "scaffold", "generator", "metric", "mean", "CI"]].copy()
df_out = df_out.sort_values(["receptor", "split", "scaffold", "metric", "mean"], ascending=[True, True, True, True, False])

df_out.head(20)


,receptor,split,scaffold,generator,metric,mean,CI
41,Glucocorticoid_receptor,dis,csk,DrugEx_RNN_epsilon_0.6,ASER,0.017470,"[0.0172, 0.0177]"
26,Glucocorticoid_receptor,dis,csk,GB_GA_mut_r_0.01,ASER,0.015722,"[0.0155, 0.0159]"
152,Glucocorticoid_receptor,dis,csk,GB_GA_mut_r_0.5,ASER,0.015579,"[0.0154, 0.0158]"
209,Glucocorticoid_receptor,dis,csk,DrugEx_GT_epsilon_0.6,ASER,0.007888,"[0.0077, 0.0080]"
29,Glucocorticoid_receptor,dis,csk,DrugEx_RNN_epsilon_0.1,ASER,0.005046,"[0.0049, 0.0052]"
50,Glucocorticoid_receptor,dis,csk,DrugEx_GT_epsilon_0.1,ASER,0.004918,"[0.0048, 0.0050]"
155,Glucocorticoid_receptor,dis,csk,Molpher,ASER,0.004595,"[0.0045, 0.0047]"
143,Glucocorticoid_receptor,dis,csk,REINVENT,ASER,0.004353,"[0.0043, 0.0045]"
59,Glucocorticoid_receptor,dis,csk,GB_GA_log_p_mut_r_0.5,ASER,0.004283,"[0.0042, 0.0044]"
161,Glucocorticoid_receptor,dis,csk,GB_GA_log_p_mut_r_0.01,ASER,0.003135,"[0.0030, 0.0032]"


You may also want a **wide (pivoted) table** for quick comparison.

In [18]:
wide = df.pivot_table(
    index=["receptor", "split", "scaffold", "generator"],
    columns="metric",
    values=["mean", "ci_low", "ci_high"],
    aggfunc="first",
)

wide.head(10)


ci_high  \
metric                                                             ASER   
receptor                split scaffold generator                          
Glucocorticoid_receptor dis   csk      DrugEx_GT_epsilon_0.1   0.005037   
                                       DrugEx_GT_epsilon_0.6   0.008044   
                                       DrugEx_RNN_epsilon_0.1  0.005159   
                                       DrugEx_RNN_epsilon_0.6  0.017684   
                                       GB_GA_log_p_mut_r_0.01  0.003229   
                                       GB_GA_log_p_mut_r_0.5   0.004401   
                                       GB_GA_mut_r_0.01        0.015926   
                                       GB_GA_mut_r_0.5         0.015800   
                                       Molpher                 0.004722   
                                       REINVENT                0.004463   

                                                                         \
metric                                                               RS   
receptor                split scaffold generator                          
Glucocorticoid_receptor dis   csk      DrugEx_GT_epsilon_0.1   0.486303   
                                       DrugEx_GT_epsilon_0.6   0.630435   
                                       DrugEx_RNN_epsilon_0.1  0.337807   
                                       DrugEx_RNN_epsilon_0.6  0.566166   
                                       GB_GA_log_p_mut_r_0.01  0.444478   
                                       GB_GA_log_p_mut_r_0.5   0.398592   
                                       GB_GA_mut_r_0.01        0.646372   
                                       GB_GA_mut_r_0.5         0.671499   
                                       Molpher                 0.472170   
                                       REINVENT                0.376265   

                                                                         \
metric                                                              SED   
receptor                split scaffold generator                          
Glucocorticoid_receptor dis   csk      DrugEx_GT_epsilon_0.1   0.204457   
                                       DrugEx_GT_epsilon_0.6   0.327682   
                                       DrugEx_RNN_epsilon_0.1  0.119352   
                                       DrugEx_RNN_epsilon_0.6  0.125052   
                                       GB_GA_log_p_mut_r_0.01  0.228572   
                                       GB_GA_log_p_mut_r_0.5   0.169924   
                                       GB_GA_mut_r_0.01        0.085139   
                                       GB_GA_mut_r_0.5         0.069313   
                                       Molpher                 0.213891   
                                       REINVENT                0.150937   

                                                                 ci_low  \
metric                                                             ASER   
receptor                split scaffold generator                          
Glucocorticoid_receptor dis   csk      DrugEx_GT_epsilon_0.1   0.004788   
                                       DrugEx_GT_epsilon_0.6   0.007729   
                                       DrugEx_RNN_epsilon_0.1  0.004909   
                                       DrugEx_RNN_epsilon_0.6  0.017240   
                                       GB_GA_log_p_mut_r_0.01  0.003026   
                                       GB_GA_log_p_mut_r_0.5   0.004161   
                                       GB_GA_mut_r_0.01        0.015532   
                                       GB_GA_mut_r_0.5         0.015367   
                                       Molpher                 0.004482   
                                       REINVENT                0.004251   

                                                                         \
metric                                                               RS   
receptor  

## 4) Pairwise significance testing across all settings

In this step, we perform bootstrap-based pairwise comparisons between all generators across all experimental settings.

For each receptor, data split, scaffold type, and metric (RS, SED, ASER), we:

- load the saved bootstrap distributions of the metric for each generator,
- compute the difference Δ = A − B for every bootstrap replicate,
- estimate the mean difference,
- compute percentile confidence intervals (CI),
- estimate a two-sided empirical p-value based on the sign of Δ.

All pairwise comparisons are aggregated into a single table (`df_tests_all`) that contains results for every generator pair across all settings. This table is later used to derive global effect size thresholds.

This approach ensures that statistical comparisons are fully consistent with the bootstrap procedure used to estimate the metrics.


In [26]:
# ============================================================
# 4) Pairwise significance testing — ALL SETTINGS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import itertools

RESULTS_DIR = Path("results_bootstrap_ci")
ALPHA = 0.05
METRICS = ["RS", "SED", "ASER"]


def load_bootstrap_vector(job_dir: Path, metric: str):
    p = job_dir / "bootstrap_samples" / f"{metric}.npy"
    if not p.exists():
        return None
    return np.load(p)


def pairwise_from_vectors(vectors: dict, alpha=0.05):
    gens = sorted(vectors.keys())
    rows = []

    for a, b in itertools.combinations(gens, 2):
        va, vb = vectors[a], vectors[b]
        if va is None or vb is None:
            continue

        d = va - vb

        rows.append({
            "A": a,
            "B": b,
            "delta_mean": float(d.mean()),
            "delta_ci_low": float(np.quantile(d, alpha/2)),
            "delta_ci_high": float(np.quantile(d, 1 - alpha/2)),
            "p_value_two_sided": float(
                2 * min((d <= 0).mean(), (d >= 0).mean())
            ),
            "abs_delta_mean": float(abs(d.mean()))
        })

    return pd.DataFrame(rows)


rows_all = []

# Structure:
# RESULTS_DIR/receptor/<scaffold>_scaffolds/split/generator/

for receptor_dir in RESULTS_DIR.iterdir():
    if not receptor_dir.is_dir():
        continue
    receptor = receptor_dir.name

    for scaffold_dir in receptor_dir.iterdir():
        if not scaffold_dir.is_dir():
            continue
        scaffold = scaffold_dir.name.replace("_scaffolds", "")

        for split_dir in scaffold_dir.iterdir():
            if not split_dir.is_dir():
                continue
            split = split_dir.name

            gen_dirs = [d for d in split_dir.iterdir() if d.is_dir()]
            if len(gen_dirs) < 2:
                continue

            for metric in METRICS:

                vectors = {}
                for gd in gen_dirs:
                    vec = load_bootstrap_vector(gd, metric)
                    if vec is not None:
                        vectors[gd.name] = vec

                if len(vectors) < 2:
                    continue

                df_pairs = pairwise_from_vectors(vectors, alpha=ALPHA)

                if df_pairs.empty:
                    continue

                df_pairs.insert(0, "receptor", receptor)
                df_pairs.insert(1, "split", split)
                df_pairs.insert(2, "scaffold", scaffold)
                df_pairs.insert(3, "metric", metric)

                rows_all.append(df_pairs)

df_tests_all = pd.concat(rows_all, ignore_index=True)

print("Total pairwise comparisons:", len(df_tests_all))
df_tests_all.head()


Total pairwise comparisons: 1320


,receptor,split,scaffold,metric,A,B,delta_mean,delta_ci_low,delta_ci_high,p_value_two_sided,abs_delta_mean
0,Glucocorticoid_receptor,dis,murcko,RS,DrugEx_GT_epsilon_0.1,DrugEx_GT_epsilon_0.6,-0.118606,-0.144618,-0.094172,0.0,0.118606
1,Glucocorticoid_receptor,dis,murcko,RS,DrugEx_GT_epsilon_0.1,DrugEx_RNN_epsilon_0.1,0.078654,0.061724,0.095402,0.0,0.078654
2,Glucocorticoid_receptor,dis,murcko,RS,DrugEx_GT_epsilon_0.1,DrugEx_RNN_epsilon_0.6,-0.111973,-0.138074,-0.086953,0.0,0.111973
3,Glucocorticoid_receptor,dis,murcko,RS,DrugEx_GT_epsilon_0.1,GB_GA_log_p_mut_r_0.01,0.044323,0.022830,0.066678,0.0,0.044323
4,Glucocorticoid_receptor,dis,murcko,RS,DrugEx_GT_epsilon_0.1,GB_GA_log_p_mut_r_0.5,0.046648,0.025516,0.066035,0.0,0.046648


## 5) Effect size interpretation thresholds (data-driven)

Here we derive thresholds for interpreting the magnitude of differences between generators.

Instead of using predefined values, thresholds are computed empirically from the distribution of absolute pairwise differences |Δ| obtained in Step 4.

For each metric (RS, SED, ASER) and scaffold type (CSK, Murcko), we compute:

- the 25th percentile → boundary for trivial differences,
- the 50th percentile (median) → boundary for small differences,
- the 75th percentile → boundary for moderate differences.

Differences larger than the 75th percentile are classified as large.

The resulting table provides data-driven categories:

- Trivial (≤25%)
- Small (25–50%)
- Moderate (50–75%)
- Large (>75%)

These thresholds summarize the empirical variability of generator performance across all experiments and are independent of statistical significance.


In [29]:
# ============================================================
# 5) Effect size interpretation table (computed from data)
# ============================================================

import numpy as np
import pandas as pd


def compute_effect_thresholds(df, metric, scaffold):
    sub = df[
        (df["metric"] == metric) &
        (df["scaffold"] == scaffold)
    ]

    deltas = sub["abs_delta_mean"].dropna().values

    if deltas.size == 0:
        return None

    t1, t2, t3 = np.quantile(deltas, [0.25, 0.50, 0.75])
    return t1, t2, t3


rows = []

for metric in ["RS", "SED", "ASER"]:
    for scaffold in ["csk", "murcko"]:

        thr = compute_effect_thresholds(
            df_tests_all,
            metric,
            scaffold
        )

        if thr is None:
            rows.append({
                "Metric": metric,
                "Scaffold": scaffold,
                "Trivial Δ (≤25%)": "NA",
                "Small Δ (25–50%)": "NA",
                "Moderate Δ (50–75%)": "NA",
                "Large Δ (>75%)": "NA",
            })
            continue

        t1, t2, t3 = thr

        rows.append({
            "Metric": metric,
            "Scaffold": scaffold.upper() if scaffold == "csk" else "Murcko",
            "Trivial Δ (≤25%)": f"Δ ≤ {t1:.3f}",
            "Small Δ (25–50%)": f"{t1:.3f} < Δ ≤ {t2:.3f}",
            "Moderate Δ (50–75%)": f"{t2:.3f} < Δ ≤ {t3:.3f}",
            "Large Δ (>75%)": f"Δ > {t3:.3f}",
        })

df_effect_table = pd.DataFrame(rows)
out_path = "effect_size_thresholds.csv"
df_effect_table.to_csv(out_path, index=False)

print("Saved to:", out_path)
df_effect_table


Saved to: effect_size_thresholds.csv


,Metric,Scaffold,Trivial Δ (≤25%),Small Δ (25–50%),Moderate Δ (50–75%),Large Δ (>75%)
0,RS,CSK,Δ ≤ 0.072,0.072 < Δ ≤ 0.189,0.189 < Δ ≤ 0.332,Δ > 0.332
1,RS,Murcko,Δ ≤ 0.044,0.044 < Δ ≤ 0.136,0.136 < Δ ≤ 0.212,Δ > 0.212
2,SED,CSK,Δ ≤ 0.045,0.045 < Δ ≤ 0.090,0.090 < Δ ≤ 0.159,Δ > 0.159
3,SED,Murcko,Δ ≤ 0.076,0.076 < Δ ≤ 0.175,0.175 < Δ ≤ 0.299,Δ > 0.299
4,ASER,CSK,Δ ≤ 0.004,0.004 < Δ ≤ 0.011,0.011 < Δ ≤ 0.018,Δ > 0.018
5,ASER,Murcko,Δ ≤ 0.001,0.001 < Δ ≤ 0.004,0.004 < Δ ≤ 0.007,Δ > 0.007
